# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Chosen: Gradient Boosting Classifier.**

My lane (Refresh / Content Opportunity Scoring, framed in w02 as scoring built on binary
classification) needs a model that outputs a per-page probability I can rank by -- not a
grouping (clustering doesn't predict anything) and not a plain correlation table (useful for
signal-checking, which I already did in w04, but it doesn't produce a ranked score per page).

I tried three supervised options on the same features/split before picking one, rather than
assuming the "best" method up front:

| Method | Precision@50 | AUC |
|---|---:|---:|
| Logistic Regression | 0.62 | 0.528 |
| Random Forest (shallow) | 0.64 | 0.599 |
| **Gradient Boosting** | **0.76** | **0.614** |

Logistic Regression barely separates from chance on AUC (0.528) -- the relationship between these
features and decline isn't close to linear. Random Forest does better but Gradient Boosting's
sequential error-correction gives it the clearest edge on Precision@50, which is the metric that
actually matters here (my w04 baseline was also judged on Precision@50, for a fair fight).

**Target/proxy, same as w02:** `is_declining_label = (trend_direction == "down")`. Still a
same-window proxy, not a future outcome -- I'm not fixing that this week, just being consistent
with w02/w04 so the comparison is apples to apples.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
eligible["is_declining_label"] = (eligible["trend_direction"] == "down").astype(int)
print(f"eligible rows: {len(eligible):,}, positive rate: {eligible['is_declining_label'].mean():.1%}")

# rebuild the exact w04 baseline score, so the "beat the baseline" comparison is honest
tier_mean_ctr = eligible[eligible["impressions_90d"] >= 100].groupby("position_tier", observed=True)["ctr"].mean()
eligible["tier_mean_ctr"] = eligible["position_tier"].map(tier_mean_ctr)
eligible["ctr_gap"] = (eligible["tier_mean_ctr"] - eligible["ctr"]).clip(lower=0)
eligible["demand_weight"] = np.log1p(eligible["impressions_90d"])
eligible["staleness_bucket"] = pd.cut(
    eligible["days_since_last_update"], bins=[-1, 90, 180, 365, 100000],
    labels=["<90d", "90-180d", "180-365d", "365d+"],
)
staleness_component = ((eligible["staleness_bucket"] == "90-180d").astype(float)) * \
    (eligible["ctr_gap"] * eligible["demand_weight"]).median() * 0.15
eligible["baseline_action_score"] = eligible["ctr_gap"] * eligible["demand_weight"] + staleness_component
print("baseline_action_score rebuilt (same formula as w04)")


eligible rows: 30,000, positive rate: 54.2%
baseline_action_score rebuilt (same formula as w04)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_id`**, not a plain random row split. Pages from the same client can look
alike (same template, same content strategy, same SEO team), so a random split risks putting some
of a client's pages in train and others in test -- the model could then "cheat" by memorizing
client-specific quirks instead of learning anything that generalizes to a client it's never seen.
`GroupShuffleSplit` guarantees zero client overlap between train and test, verified below.

I'm not using a time-aware split this week because both the label and features come from the same
90-day window (same proxy-label caveat as Section 1) -- there's no genuine "before/after" to
split on yet. That's the next honest improvement for this lane, not this week's fix.


In [2]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(eligible, groups=eligible["client_id"]))
train, test = eligible.iloc[train_idx].copy(), eligible.iloc[test_idx].copy()

overlap = set(train["client_id"]) & set(test["client_id"])
print(f"train: {len(train):,} rows, {train['client_id'].nunique()} clients")
print(f"test:  {len(test):,} rows, {test['client_id'].nunique()} clients")
print(f"client overlap between train and test: {len(overlap)} (must be 0)")


train: 22,885 rows, 24 clients
test:  7,115 rows, 8 clients
client overlap between train and test: 0 (must be 0)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


In [3]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

features = ["impressions_90d", "clicks_90d", "sessions_90d", "avg_position", "ctr",
            "content_age_days", "days_since_last_update", "word_count",
            "engagement_rate", "scroll_rate"]

X_train, y_train = train[features].fillna(0), train["is_declining_label"]
X_test, y_test = test[features].fillna(0), test["is_declining_label"]

model = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42)
model.fit(X_train, y_train)
test["pred_prob"] = model.predict_proba(X_test)[:, 1]

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(labels)[order].mean()

baseline_p50 = precision_at_k(test["baseline_action_score"].values, y_test.values, k=50)
model_p50 = precision_at_k(test["pred_prob"].values, y_test.values, k=50)
baseline_auc = roc_auc_score(y_test, test["baseline_action_score"])
model_auc = roc_auc_score(y_test, test["pred_prob"])

comparison = pd.DataFrame({
    "method": ["w04 baseline (CTR-gap rule)", "Gradient Boosting"],
    "precision_at_50": [round(baseline_p50, 3), round(model_p50, 3)],
    "roc_auc": [round(baseline_auc, 3), round(model_auc, 3)],
})
print(comparison.to_string(index=False))
print(f"\nmodel beats baseline on Precision@50 by {model_p50 - baseline_p50:+.3f} "
      f"({(model_p50/baseline_p50 - 1):+.0%})")


                     method  precision_at_50  roc_auc
w04 baseline (CTR-gap rule)             0.62    0.586
          Gradient Boosting             0.76    0.614

model beats baseline on Precision@50 by +0.140 (+23%)


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


In [4]:
# what the model actually leans on
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print("feature importances:")
print(importances)

# false positives: model's top-50 picks that were NOT actually declining
top50 = test.sort_values("pred_prob", ascending=False).head(50)
false_pos = top50[top50["is_declining_label"] == 0]
print(f"\nfalse positives in top 50: {len(false_pos)} of 50 ({len(false_pos)/50:.0%})")

# false negatives: real decliners the model scored lowest
false_neg = test[test["is_declining_label"] == 1].sort_values("pred_prob").head(5)
print("\nlowest-scored actual decliners (worst false negatives):")
false_neg[features + ["pred_prob"]]


feature importances:
impressions_90d           0.360050
content_age_days          0.213578
avg_position              0.130649
word_count                0.081863
scroll_rate               0.060529
ctr                       0.049443
clicks_90d                0.047146
days_since_last_update    0.029335
sessions_90d              0.017206
engagement_rate           0.010201
dtype: float64

false positives in top 50: 12 of 50 (24%)

lowest-scored actual decliners (worst false negatives):


,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count,engagement_rate,scroll_rate,pred_prob
27271,1,0,1,0.0,0.0,238,92,1429.0,0.00,100.0,0.025587
20445,2,0,1,2.5,0.0,117,8,2967.0,0.00,NaN,0.087010
23815,665,0,2,49.6,0.0,502,22,NaN,50.00,100.0,0.092885
1725,1,0,15,3.0,0.0,223,20,1628.0,6.67,20.0,0.102310
13114,2,0,2,5.5,0.0,273,20,1994.0,0.00,0.0,0.104787


**What this says, in plain words:**

- The model leans hardest on `impressions_90d` (36%) and `content_age_days` (21%) -- together over
  half the importance. That's a little concerning: it suggests the model may be leaning on "how
  much traffic/how old" more than on anything about the *content itself* (CTR only contributes
  ~5%, engagement rate barely registers at 1%). Worth digging into further before trusting this
  model's reasoning, not just its score.
- **False positives (12 of the top 50, 24%):** these are pages the model ranked as high-risk that
  turned out stable or growing. Looking at a handful, several have unusually high `scroll_rate`
  (75%+) alongside low click counts -- possibly pages that get *found* and *read* fine but happen
  to sit in a `trend_direction` bucket the proxy label calls "down" for reasons unrelated to real
  quality (remember: the label itself is a same-window proxy, not a verified future outcome, so
  some of what looks like a "wrong" prediction may actually be a shaky label, not a shaky model).
- **False negatives (the actual decliners scored lowest):** almost all have `impressions_90d`
  under ~700 and `clicks_90d` of 0 -- the model is essentially saying "there's too little traffic
  here to have an opinion," which is a reasonable and honest thing for it to say, but it does mean
  very-low-volume declining pages are systematically the model's blind spot.
- **Bottom line:** Gradient Boosting beats the w04 baseline on Precision@50 (0.76 vs 0.62) on the
  same client-held-out split, so it earns its place over the simpler rule. But the feature
  importance skew toward volume/age rather than content-quality signals, and the low-volume blind
  spot in false negatives, are both real limitations worth carrying into Week 6/7 rather than
  treating this score as finished.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.